# 🏥 Clinic Patient Admissions Analysis

**Author:** [Your Name]  
**Date:** May 2026  
**Dataset:** Clinic Patient Admissions (Dec 2025 – Jan 2026)

---

## Overview

This notebook analyzes a clinic dataset containing patient admission records across December 2025 and January 2026. The analysis focuses on:

1. **Dataset Overview** – Loading and exploring the data structure
2. **Peak Admission Days** – Identifying the busiest days in January 2026
3. **Doctor Workload Analysis** – Ranking the busiest doctors in January 2026
4. **Month-over-Month Comparison** – Comparing doctor activity: Dec 2025 vs Jan 2026
5. **Billing Analysis** – Total billing amount per doctor

**Key Findings:**
- Jan 2nd and Jan 22nd were the busiest admission days in January 2026 (2 patients each)
- Dr. Robert Chen was the busiest doctor in January 2026 (3 patients)
- Dr. Alan Patel and Dr. Robert Chen both saw more patients in January compared to December
- Dr. Sarah Jenkins led overall billing revenue


---
## 1. Setup & Data Loading


In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set consistent plot style
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully.')

In [ ]:
# ---------------------------------------------------------------
# Load dataset
# If running in Google Colab, upload 'clinic dataset.xlsx' first.
# ---------------------------------------------------------------
from google.colab import files

uploaded = files.upload()  # Upload 'clinic dataset.xlsx'
df = pd.read_excel('clinic dataset.xlsx')

# Parse dates
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'])
df['Discharge_Date'] = pd.to_datetime(df['Discharge_Date'])

print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Date range: {df["Admission_Date"].min().date()} to {df["Admission_Date"].max().date()}')

---
## 2. Dataset Overview


In [ ]:
# Preview the first few rows
df.head()

In [ ]:
# Summary statistics
df.describe(include='all')

---
## 3. Peak Admission Days — January 2026

Which days had the most patient admissions in January 2026?


In [ ]:
# Filter for January 2026
january_2026_df = df[(df['Admission_Date'].dt.year == 2026) & (df['Admission_Date'].dt.month == 1)]

# Group by date and count patients
daily_patients = (
    january_2026_df
    .groupby('Admission_Date')
    .size()
    .reset_index(name='Number_of_Patients')
    .sort_values('Number_of_Patients', ascending=False)
    .head(5)
)

print('Top 5 days with most patient admissions in January 2026:')
display(daily_patients)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    x='Admission_Date', y='Number_of_Patients',
    data=daily_patients, palette='viridis',
    hue='Admission_Date', legend=False, ax=ax
)
ax.set_xlabel('Admission Date')
ax.set_ylabel('Number of Patients')
ax.set_title('Top 5 Busiest Admission Days — January 2026')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
## 4. Busiest Doctors — January 2026

Which doctors handled the most patients in January 2026?


In [ ]:
# Count patients per doctor in January 2026
january_2026_admissions = df[(df['Admission_Date'].dt.year == 2026) & (df['Admission_Date'].dt.month == 1)]

top_5_doctors = (
    january_2026_admissions['Doctor_Name']
    .value_counts()
    .reset_index()
    .rename(columns={'count': 'Number_of_Patients_Treated'})
    .head(5)
)

print('Top 5 busiest doctors in January 2026:')
display(top_5_doctors)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    x='Doctor_Name', y='Number_of_Patients_Treated',
    data=top_5_doctors, palette='cubehelix',
    hue='Doctor_Name', legend=False, ax=ax
)
ax.set_xlabel('Doctor Name')
ax.set_ylabel('Number of Patients Treated')
ax.set_title('Top 5 Busiest Doctors — January 2026')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

---
## 5. Month-over-Month Doctor Comparison — Dec 2025 vs Jan 2026

Which doctors saw an increase in patient load from December 2025 to January 2026?


In [ ]:
# Count patients per doctor for each month
december_2025_admissions = df[(df['Admission_Date'].dt.year == 2025) & (df['Admission_Date'].dt.month == 12)]

dec_counts = (
    december_2025_admissions['Doctor_Name']
    .value_counts()
    .reset_index()
    .rename(columns={'count': 'Patients_Dec_2025'})
)

jan_counts = (
    january_2026_admissions['Doctor_Name']
    .value_counts()
    .reset_index()
    .rename(columns={'count': 'Patients_Jan_2026'})
)

# Merge and compute difference
combined = (
    pd.merge(dec_counts, jan_counts, on='Doctor_Name', how='outer')
    .fillna(0)
    .astype({'Patients_Dec_2025': int, 'Patients_Jan_2026': int})
)
combined['Difference'] = combined['Patients_Jan_2026'] - combined['Patients_Dec_2025']
combined = combined.sort_values('Difference', ascending=False)

print('Doctor patient counts — Dec 2025 vs Jan 2026:')
display(combined)

In [ ]:
# Doctors with increased patient load
increased = combined[combined['Difference'] > 0]

if not increased.empty:
    melted = increased.melt(
        id_vars='Doctor_Name',
        value_vars=['Patients_Dec_2025', 'Patients_Jan_2026'],
        var_name='Month', value_name='Number_of_Patients'
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(
        x='Doctor_Name', y='Number_of_Patients', hue='Month',
        data=melted,
        palette={'Patients_Dec_2025': 'skyblue', 'Patients_Jan_2026': 'lightcoral'},
        ax=ax
    )
    ax.set_xlabel('Doctor Name')
    ax.set_ylabel('Number of Patients')
    ax.set_title('Doctors with Increased Patient Load: Jan 2026 vs Dec 2025')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(title='Month')
    plt.tight_layout()
    plt.show()
else:
    print('No doctors had an increased patient load in January 2026.')

---
## 6. Total Billing Amount per Doctor

Which doctors generated the most billing revenue across the full dataset?


In [ ]:
doctor_billing = (
    df.groupby('Doctor_Name')['Billing_Amount']
    .sum()
    .reset_index()
    .sort_values('Billing_Amount', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    x='Doctor_Name', y='Billing_Amount',
    data=doctor_billing, palette='magma',
    hue='Doctor_Name', legend=False, ax=ax
)
ax.set_xlabel('Doctor Name')
ax.set_ylabel('Total Billing Amount ($)')
ax.set_title('Total Billing Revenue per Doctor')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

display(doctor_billing)

---
## 7. Summary & Key Takeaways

| Insight | Finding |
|---|---|
| Busiest admission days (Jan 2026) | Jan 2nd and Jan 22nd (2 patients each) |
| Most active doctor (Jan 2026) | Dr. Robert Chen (3 patients) |
| Doctors with increased workload (Jan vs Dec) | Dr. Alan Patel (+1), Dr. Robert Chen (+1) |
| Highest billing revenue | Dr. Sarah Jenkins |

**Next Steps (suggested):**
- Expand the dataset to cover more months for trend analysis
- Add patient outcome metrics to correlate with billing and doctor workload
- Explore department-level admission patterns
